In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
import operator

create llms

In [2]:
generator_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")
evaluator_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")
optimizer_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct")

create schema

In [4]:
class TweetEvaluation(BaseModel):
    evaluation : Literal["Approved","Needs_Improvement"]=Field (...,description="Final Evaluation Result")
    feedback:str=Field(..., description="Feedback for the tweet")

In [5]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

create state

In [6]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal["Approved","Needs_Improvement"]
    feedback:str
    iteration:int
    max_iteration:int
    tweet_histor:Annotated[list[str],operator.add]
    feedback_history:Annotated[list[str],operator.add]


In [7]:
def generate_tweet(state:TweetState):
    #prompt
    messages=[
        SystemMessage(content="You are a funny and clever Tweeter/X influencer"),
        HumanMessage(content=f"""Write a short, original and hilarious tweet on topic: "{state['topic']}"
        Rules:
                     - Do not use question question-answer
                     - Max 280 characters
                     - Use observational humour,irony, sarcasm or cultured references
                     - Think in meme logic, punchlines or relatable  takes
                     - Use simple, day to day english
""")
    ]
    #send generator llm
    response = generator_llm.invoke(messages).content
    #return
    return{'tweet':response, 'tweet_history':[response]}

In [8]:
def evaluate_tweet(state:TweetState):
    #prompt
    messages=[
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic, You evaluate tweets based on humour , originality, virality and tweet format"),
        HumanMessage(content=f""" Evaluate the following tweet:
                     Tweet:"{state['tweet']}"
                     Use the criteria is below to evaluate the tweet:
                     1. Originality - Fresh, not should be seen 100 times before
                     2. Humor - Genuinely make smile and laugh
                     3. Punchlines - It is short, sharp and scroll stopping
                     4. Virality Potential - People should retweet or share it
                     5. Formal - It is well formal tweet

                     Auto-reject if:
                     - its written in question - answer format
                     - It exceeds 280 characters 
                     - It reads like a traditional setup-punchline joke
                     - Don't end with generic, deflating lines

                    ### Respond only in structured format
                    evaluation
                    feedback

""")
    ]
    response = structured_evaluator_llm.invoke(messages)
    return {'evaluation':response.evaluation,'feedback':response.feedback, 'feedback_history':[response.feedback]}

In [9]:
def optimize_tweet(state:TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on a given feedback"),
        HumanMessage(content=f"""Improve the tweet based on this 
                     feedback:"{state['feedback']}",
                     topic:"{state['topic']},
                     original_tweet: {state['tweet']}
                     Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters""")
    ]
    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration']
    return {'tweet':response, 'iteration':iteration, 'tweet_history':[response]}

In [ ]:
def route_evaluation(state:TweetState)->Literal["Approved","Needs_Improvement"]:
    if state['evaluation']=='Approved' or state['iteration']>=state['max_iteration']:
        return "Approved"
    else:
        return "Needs_Improvement"


In [12]:
graph = StateGraph(TweetState)
graph.add_node('Generate', generate_tweet)
graph.add_node("Evaluate",evaluate_tweet)
graph.add_node("Optimize",optimize_tweet)

graph.add_edge(START, 'Generate')
graph.add_edge("Generate","Evaluate")
graph.add_conditional_edges("Evaluate",route_evaluation,{"Approved":END, "Needs_Improvement":"Optimize"})
graph.add_edge("Optimize","Evaluate")




In [13]:
workflow= graph.compile()

In [14]:
initial_state = {
    "topic":"IT",
    "iteration":1,
    "max_iteration":5
}

In [17]:
final_state = workflow.invoke(initial_state)

In [18]:
print(final_state)

{'topic': 'IT', 'tweet': '"Just spent 2 hours on hold with IT. I\'m starting to think their \'support\' team is actually just a group of interns playing \'Don\'t Starve\' on their work laptops. \'The server is down\' = \'I fell asleep at my desk\'" #ITLife #SupportNightmares', 'evaluation': 'Approved', 'feedback': "The tweet excels in originality with a fresh take on the common frustration with IT support. It effectively uses humor by creatively imagining IT support teams as interns playing games on their work laptops. The punchlines are sharp and likely to make readers smile and laugh. The tweet's format is well-structured, concise, and adheres to the 280-character limit. It also ends on a strong, relatable note that avoids generic deflation. Overall, it has high virality potential due to its relatability and humor.", 'iteration': 1, 'max_iteration': 5, 'tweet_histor': [], 'feedback_history': ["The tweet excels in originality with a fresh take on the common frustration with IT support